# MRP / 06 — Continuation after large bars: test window 2023-07-01 – 2025-12-31 (one look)

Protocol: [`MRP 004`](../../discussions/MNQ_Return_Prediction/004_claude_mrp03_continuation_preregistration.md). Trade **with** the large bar (|z| ≥ k), 8 fixed configurations: k ∈ {3, 4} × d ∈ {0, 1} × h ∈ {3, 6}, all sessions. Ticks per contract (1 tick = $0.50).

Gates: **C1** n ≥ 75 · **C2** net > 0 and day-block bootstrap 95% lower bound of total > 0 · **C3** net > 0 in ≥ 2 of 3 periods · **C4** Holm-adjusted one-sided p ≤ 0.05 across the 8.

Runs once: it refuses if `06/summary.json` exists. Set your Lucid commission first.

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("MRP_SYNTHETIC") == "1"
if SYNTHETIC:   # separate folder: the reversal smoke data has a planted reversal and no FX effect
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("MRP_REV_SYNTH_DIR") or tempfile.mkdtemp(prefix="mrp_rev_")
    os.environ["MRP_REV_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]
    os.environ.pop("DUKASCOPY_DIR", None)

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import mnq_predict as P
from equity_trading import reversal as V

CFG = P.SYNTH_CFG if SYNTHETIC else P.MRPConfig()
D = dt.date
OUT = Path("06"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
print("engines", P.MRP_VERSION, V.REV_VERSION, "| synthetic" if SYNTHETIC else "| real data")
if SYNTHETIC and not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
    P.write_synthetic(os.environ["EQUITY_TRADING_DIR"], CFG.warmup_start, CFG.test[1], CFG, beta_jpy=0.0,
                      reversal_points=-6.0, tails_df=4)


In [ ]:
COMMISSION_PER_SIDE = 0.50          # USD per contract per side (Lucid)
COMMISSION_CONFIRMED = SYNTHETIC
if (OUT / "summary.json").exists() and not SYNTHETIC:
    raise RuntimeError("06 has already been run: the test window is one look (MRP 004 §6).")
if not COMMISSION_CONFIRMED:
    raise RuntimeError("Confirm the Lucid commission and set COMMISSION_CONFIRMED = True.")
RC = V.RevConfig(commission_per_side=COMMISSION_PER_SIDE)
if SYNTHETIC:
    PERIODS = {"2021Q1": (D(2021, 1, 1), D(2021, 3, 31)), "2021Q2": (D(2021, 4, 1), D(2021, 6, 30))}
else:
    PERIODS = {"2023H2": (D(2023, 7, 1), D(2023, 12, 31)), "2024": (D(2024, 1, 1), D(2024, 12, 31)),
               "2025": (D(2025, 1, 1), D(2025, 12, 31))}
grid = P.load_grid(CFG.warmup_start, CFG.test[1], CFG)
sig2 = V.event_sigma(grid, None, CFG.refits_test, CFG.test[1], CFG)
bt = V.bar_table(grid, sig2, CFG)
bt = bt[(bt["trade_date"] >= CFG.test[0]) & (bt["trade_date"] <= CFG.test[1])]
print(len(bt), "bars | market cost", RC.market_cost_ticks, "ticks | realized", RC.realized_cost_ticks, "ticks")

In [ ]:
res, store = V.continuation_test(bt, RC, PERIODS)
cols = ["config", "n", "gross", "net", "t", "p_holm", "boot_lower_total", "net_realized", "gross_long", "gross_short",
        "win_rate", "mean_win", "mean_loss", "q05", "q01", "worst", "max_dd", "periods_positive"] + \
       [f"net_{p}" for p in PERIODS] + ["C1", "C2", "C3", "C4", "confirmed"]
display(res[cols].round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
for c, tr in store.items():
    ax[0].plot(pd.to_datetime(tr["trade_date"]), (tr["gross_ticks"] - RC.market_cost_ticks).cumsum(), label=c)
ax[0].axhline(0, color="k", lw=.6); ax[0].legend(fontsize=7); ax[0].set_title("cumulative net ticks per contract (market cost)")
best = res.sort_values("net", ascending=False)["config"].iloc[0]
ax[1].hist(np.clip(store[best]["gross_ticks"] - RC.market_cost_ticks, -600, 600), bins=80)
ax[1].axvline(0, color="k"); ax[1].set_title(f"net ticks per trade: {best}")
plt.tight_layout(); plt.savefig(OUT / "continuation.png", dpi=110); plt.show()
summary = {"engines": [P.MRP_VERSION, V.REV_VERSION], "synthetic": SYNTHETIC, "commission_per_side": COMMISSION_PER_SIDE,
           "market_cost_ticks": RC.market_cost_ticks, "results": res.to_dict("records"),
           "confirmed": res.loc[res["confirmed"], "config"].tolist()}
(OUT / "summary.json").write_text(json.dumps(summary, indent=1, default=lambda o: bool(o) if isinstance(o, np.bool_) else float(o)))
print("confirmed:", summary["confirmed"])